# Sample-Based Krylov Quantum Diagonalization (SKQD)

**Sample-Based Krylov Quantum Diagonalization (SKQD)** is a hybrid quantum-classical algorithm that combines the theoretical guarantees of Krylov Quantum Diagonalization (KQD) with the practical efficiency of sample-based methods. Instead of expensive quantum measurements to compute Hamiltonian matrix elements, SKQD samples from quantum states to construct a computational subspace, then diagonalizes the Hamiltonian within that subspace classically.

## Why SKQD?

Traditional quantum algorithms like VQE face several fundamental challenges:

- **Optimization complexity**: VQE requires optimizing many variational parameters in a high-dimensional, non-convex landscape
- **Measurement overhead**: Computing expectation values $\langle\psi(\theta)|H|\psi(\theta)\rangle$ requires many measurements for each Pauli term
- **Barren plateaus**: Optimization landscapes can become exponentially flat, making gradient-based optimization ineffective
- **Parameter initialization**: Poor initial parameters can lead to local minima far from the global optimum

SKQD addresses these fundamental limitations:

- **No optimization required**: Uses deterministic quantum time evolution instead of variational circuits
- **Provable convergence**: Theoretical guarantees based on the Rayleigh-Ritz variational principle
- **Measurement efficient**: Only requires computational basis measurements (Z-basis), the most natural measurement on quantum hardware  
- **Noise resilient**: Can filter out problematic measurement outcomes and handle finite sampling
- **Systematic improvement**: Increasing Krylov dimension monotonically improves ground state estimates
- **Hardware friendly**: Time evolution circuits are more amenable to near-term quantum devices than deep variational ansätze




## Understanding Krylov Subspaces

### What is a Krylov Subspace?

A **Krylov subspace** $\mathcal{K}^r$ of dimension $r$ is the space spanned by vectors obtained by repeatedly applying an operator $A$ to a reference vector $|\psi\rangle$:

$$\mathcal{K}^r = \text{span}\{|\psi\rangle, A|\psi\rangle, A^2|\psi\rangle, \ldots, A^{r-1}|\psi\rangle\}$$



### The SKQD Algorithm

The key insight of SKQD is that we can:
1. **Generate** Krylov states $U^k|\psi\rangle$ using quantum time evolution
2. **Sample** from these states to get computational basis measurements
3. **Combine** all samples to form a computational subspace
4. **Diagonalize** the Hamiltonian within this subspace classically

This approach is much more efficient than computing matrix elements via quantum measurements!


In [ ]:
import time

import cudaq
import numpy as np
import pickle

from openfermion.chem import MolecularData, geometry_from_pubchem
from openfermion.transforms import get_fermion_operator, jordan_wigner
from openfermionpyscf import run_pyscf

from skqd_src.pre_and_postprocessing import *

from scipy.sparse import csr_matrix as sp_csr_matrix
from scipy.sparse.linalg import eigsh as sp_eigsh

# CuPy needs an NVIDIA GPU. Without one (e.g. the CPU Docker image) the
# post-processing falls back to NumPy/SciPy.
try:
    import cupy as cp
    HAS_GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception:
    cp, HAS_GPU = None, False

if HAS_GPU:
    from cupyx.scipy.sparse import csr_matrix as cp_csr_matrix
    from cupyx.scipy.sparse.linalg import eigsh as cp_eigsh

if cudaq.num_available_gpus() > 0 and cudaq.has_target("nvidia"):
    cudaq.set_target("nvidia")
else:
    cudaq.set_target("qpp-cpu")

cudaq.set_random_seed(42)
np.random.seed(43)
if HAS_GPU:
    cp.random.seed(44)

print("Using:", cudaq.__version__)
print("Simulator:", cudaq.get_target().name,
      "| Post-processing:", "GPU (CuPy)" if HAS_GPU else "CPU (NumPy/SciPy)")

## Problem Setup: Water (H$_2$O) in the STO-3G Basis

We demonstrate SKQD on the electronic structure Hamiltonian of a water molecule:

- **Geometry** is fetched from PubChem through OpenFermion (`geometry_from_pubchem`), so this cell needs network access.
- **Integrals** are computed with PySCF in the minimal STO-3G basis: 7 spatial orbitals, i.e. 14 spin orbitals.
- **Qubit mapping** is Jordan-Wigner, so each spin orbital becomes one qubit: **14 qubits**, 10 electrons.

$$H = \sum_{pq} h_{pq}\, a_p^\dagger a_q + \frac{1}{2}\sum_{pqrs} h_{pqrs}\, a_p^\dagger a_q^\dagger a_r a_s \;\xrightarrow{\text{JW}}\; \sum_j h_j P_j$$

The reference energy is full configuration interaction (FCI), which is exact within the basis set.

### Time evolution with qDRIFT

The Jordan-Wigner Hamiltonian has about 1,400 Pauli terms, so a single first-order Trotter step is about 1,400 Pauli exponentials (roughly 16,600 CNOTs). Instead we use qDRIFT (Campbell, PRL 123, 070503, 2019): with $\lambda = \sum_j |h_j|$, each of $N$ gates picks term $j$ with probability $|h_j|/\lambda$ and applies $e^{-i\,\mathrm{sgn}(h_j)\,\lambda t/N\, P_j}$. Large terms (mostly low-weight $Z$ and $ZZ$) are picked most often, and the cost depends on $N$, not on the number of terms.

One random sequence is drawn and reused as $U$ for every Krylov power, so the Krylov states are $U^k|\psi_0\rangle$ for one fixed unitary.

In [ ]:
geometry = geometry_from_pubchem("water")
molecule = run_pyscf(MolecularData(geometry, basis="sto-3g", multiplicity=1, charge=0),
                     run_scf=True, run_fci=True)

num_spins = molecule.n_qubits           # 14 spin orbitals -> 14 qubits
num_electrons = molecule.n_electrons    # occupied spin orbitals in the Hartree-Fock state
exact_ground_state_energy = molecule.fci_energy

if num_spins >= 63:
    raise ValueError(f"Vectorized implementation of postprocessing supports max 62 qubits due to int64 packing. Requested: {num_spins}")

shots = 100_000 if cudaq.get_target().name == "nvidia" else 1_000

total_time_evolution = np.pi
qdrift_samples = 200  # gates in the qDRIFT sequence for one application of U

max_k = 12  # largest k for U^k

H = jordan_wigner(get_fermion_operator(molecule.get_molecular_hamiltonian()))
H.compress()

# OpenFermion terms -> Pauli words, with qubit i at position i of the word
pauli_words, hamiltonian_coefficients = [], []
for term, coeff in H.terms.items():
    assert abs(coeff.imag) < 1e-6
    word = ["I"] * num_spins
    for qubit, pauli in term:
        word[qubit] = pauli
    pauli_words.append("".join(word))
    hamiltonian_coefficients.append(coeff.real)
hamiltonian_coefficients_numpy = np.array(hamiltonian_coefficients)

# The identity term only adds a global phase to the time evolution, so it is
# left out of the circuit (it stays in the Hamiltonian for post-processing).
circuit_terms = [i for i, w in enumerate(pauli_words) if set(w) != {"I"}]
circuit_coefficients = hamiltonian_coefficients_numpy[circuit_terms]

# qDRIFT: draw the gate sequence for U = exp(-iHt)
qdrift_lambda = np.abs(circuit_coefficients).sum()
qdrift_rng = np.random.default_rng(7)
qdrift_terms = qdrift_rng.choice(len(circuit_terms), size=qdrift_samples,
                                 p=np.abs(circuit_coefficients) / qdrift_lambda)
qdrift_pauli_words = [pauli_words[circuit_terms[j]] for j in qdrift_terms]
qdrift_angles = (-np.sign(circuit_coefficients[qdrift_terms])
                 * qdrift_lambda * total_time_evolution / qdrift_samples).tolist()

qdrift_cnots = sum(2 * (len(w) - w.count("I") - 1) for w in qdrift_pauli_words)

print(f"{num_spins} qubits, {num_electrons} electrons, {len(pauli_words)} Pauli terms")
print(f"HF energy:  {molecule.hf_energy:.6f} Ha")
print(f"FCI energy: {exact_ground_state_energy:.6f} Ha")
print(f"qDRIFT: lambda = {qdrift_lambda:.2f}, {qdrift_samples} gates (~{qdrift_cnots} CNOTs) per U")
print(f"shots = {shots}, max_k = {max_k}")

## Krylov State Generation via Repeated Evolution

For SKQD, we generate the Krylov sequence:
$$|\psi_0\rangle,  U|\psi_0\rangle,  U^2|\psi_0\rangle, \ldots,  U^{k-1}|\psi_0\rangle$$

where $U = e^{-iHT}$ is approximated by a qDRIFT sequence.

**Implementation Strategy**:
1. Start with the Hartree-Fock reference state $|\psi_0\rangle = |1\ldots1\,0\ldots0\rangle$ (the first 10 spin orbitals occupied)
2. Apply the qDRIFT time evolution $k$ times for $U^k|\psi_0\rangle$
3. Measure each Krylov state in computational basis
4. Accumulate measurement statistics across all Krylov powers



In [ ]:
@cudaq.kernel
def quantum_krylov_evolution_circuit(
        num_qubits: int,
        num_occupied: int,
        krylov_power: int,
        U_pauli_words: list[cudaq.pauli_word],
        U_angles: list[float]):

    """
    Generate Krylov states via repeated time evolution.

    Args:
        num_qubits: Number of qubits in the system
        num_occupied: Number of occupied spin orbitals in the reference state
        krylov_power: Power k for computing U^k|ψ⟩
        U_pauli_words: Pauli words of the qDRIFT sequence for U
        U_angles: Rotation angle of each gate in the sequence

    Returns:
        Measurement results in computational basis
    """

    qubits = cudaq.qvector(num_qubits)

    #Prepare Hartree-Fock state as reference |11...100...0⟩
    for qubit_index in range(num_occupied):
        x(qubits[qubit_index])


    for _ in range(krylov_power):                              #applies U^k where U ≈ exp(-iHt)

        for i in range(len(U_angles)):                         #applies the qDRIFT sequence for U
            exp_pauli(U_angles[i], qubits, U_pauli_words[i])


    mz(qubits)

## Quantum Measurements and Sampling


### The Sampling Process

For each Krylov power $k = 0, 1, 2, \ldots, K-1$:
1. **Prepare** the state $U^k|\psi_0\rangle$ using our quantum circuit
2. **Measure** in the computational basis many times
3. **Collect** the resulting bitstring counts

The key insight: these measurement outcomes give us a statistical representation of each Krylov state, which we can then use to construct our computational subspace classically.


In [ ]:
all_measurement_results = []

for krylov_power in range(max_k):
    t0 = time.perf_counter()

    sampling_result = cudaq.sample(
        quantum_krylov_evolution_circuit,
        num_spins,
        num_electrons,
        krylov_power,
        qdrift_pauli_words,
        qdrift_angles,
        shots_count=shots)

    all_measurement_results.append(dict(sampling_result.items()))
    print(f"Krylov state U^{krylov_power}: {len(sampling_result)} unique bitstrings "
          f"({time.perf_counter() - t0:.1f} s)")

cumulative_results = calculate_cumulative_results(all_measurement_results)

## Classical Post-Processing and Diagonalization

Now comes the classical part of SKQD: we use our quantum measurement data to construct and diagonalize the Hamiltonian within each Krylov subspace.

1. **Extract basis states** from measurement counts
2. **Project Hamiltonian** onto the computational subspace spanned by these states  
3. **Diagonalize** the projected Hamiltonian classically
4. **Extract ground state energy** estimate

### Matrix Construction Details

The core of SKQD is constructing the effective Hamiltonian matrix within the computational subspace:

1. **Computational Subspace Formation**: From quantum measurements, we obtain a set of computational basis states $\{|s_1\rangle, |s_2\rangle, \ldots, |s_d\rangle\}$ that spans our approximation to the Krylov subspace.

2. **Matrix Element Computation**: For each Pauli term $P_k$ in the Hamiltonian with coefficient $h_k$:
   $$H = \sum_k h_k P_k$$
   
   We compute matrix elements $\langle s_i | P_k | s_j \rangle$ by applying the Pauli string $P_k$ to each basis state $|s_j\rangle$.

3. **Effective Hamiltonian**: The projected Hamiltonian becomes:
   $$H_{\text{eff}}[i,j] = \sum_k h_k \langle s_i | P_k | s_j \rangle$$

We demonstrate two eigenvalue solver approaches below: an explicit CSR sparse matrix and a matrix-free Lanczos iteration.

### Approach 1: Vectorized CSR Sparse Matrix

We build the projected Hamiltonian as a CuPy CSR sparse matrix on the GPU and solve with `cupyx.scipy.sparse.linalg.eigsh`. Without a GPU, the same vectorized code runs in NumPy and the matrix is solved with `scipy.sparse.linalg.eigsh`.


In [ ]:
csr_energies = []

for k in range(1, max_k):

    cumulative_subspace_results = cumulative_results[k]
    basis_states = get_basis_states_as_array(cumulative_subspace_results, num_spins)
    subspace_dimension = len(cumulative_subspace_results)
    assert len(cumulative_subspace_results) == basis_states.shape[0]

    v0_np = np.random.random((subspace_dimension,)).astype(np.float64)
    v0_np = v0_np / np.linalg.norm(v0_np)

    matrix_rows, matrix_cols, matrix_elements = vectorized_projected_hamiltonian(
        basis_states, pauli_words, hamiltonian_coefficients_numpy, HAS_GPU)

    if HAS_GPU:
        projected_hamiltonian = cp_csr_matrix(
            (matrix_elements, (matrix_rows, matrix_cols)),
            shape=(subspace_dimension, subspace_dimension))
        eigenvalue = cp_eigsh(projected_hamiltonian, return_eigenvectors=False,
                              k=1, which='SA', v0=cp.asarray(v0_np, dtype=cp.complex128))
    else:
        projected_hamiltonian = sp_csr_matrix(
            (matrix_elements, (matrix_rows, matrix_cols)),
            shape=(subspace_dimension, subspace_dimension))
        eigenvalue = sp_eigsh(projected_hamiltonian, return_eigenvectors=False,
                              k=1, which='SA', v0=v0_np)

    csr_energies.append(float(np.min(eigenvalue).real))
    print(f"k={k}: subspace dimension {subspace_dimension}, "
          f"E = {csr_energies[-1]:.6f} Ha, E - E_FCI = {csr_energies[-1] - exact_ground_state_energy:.1e} Ha")

### Approach 2: Matrix-Free Lanczos via `distributed_eigsh`

Instead of explicitly constructing the projected Hamiltonian as a sparse matrix, we can apply the Hamiltonian as a linear operator during Lanczos iteration. This matrix-free approach avoids storing the full sparse matrix and naturally extends to multiple GPUs (demonstrated in the multi-GPU section below).

Without a GPU, `matrix_free_eigsh_cpu` applies the same bit-mask matrix-vector product with NumPy inside a SciPy `LinearOperator`.

In [ ]:
lo_energies = []

for k in range(1, max_k):

    cumulative_subspace_results = cumulative_results[k]
    basis_states = get_basis_states_as_array(cumulative_subspace_results, num_spins)
    subspace_dimension = len(cumulative_subspace_results)

    v0_np = np.random.random((subspace_dimension,)).astype(np.float64)
    v0_np = v0_np / np.linalg.norm(v0_np)

    if HAS_GPU:
        v0 = cp.asarray(v0_np, dtype=cp.complex128)

        ham_data = prepare_hamiltonian_data(basis_states, pauli_words, hamiltonian_coefficients_numpy)
        cp.get_default_memory_pool().free_all_blocks()
        ncv = min(20, subspace_dimension - 1)

        eigenvalues, = distributed_eigsh(
            ham_data, rank=0, size=1, nccl_comm_obj=None,
            k=1, which='SA', ncv=ncv, v0_full=v0,
            tile_size=32, return_eigenvectors=False)

        eigenvalue_real = float(cp.asnumpy(eigenvalues[0]).real)
    else:
        eigenvalues = matrix_free_eigsh_cpu(
            basis_states, pauli_words, hamiltonian_coefficients_numpy,
            v0=v0_np, k=1, which='SA', return_eigenvectors=False)

        eigenvalue_real = float(eigenvalues[0].real)

    lo_energies.append(eigenvalue_real)

## Results Analysis and Convergence

Let's visualize our results and analyze how SKQD converges to the true ground state energy. 

### What to Expect:

- **Monotonic improvement**: Each additional Krylov dimension should give a better (lower) energy estimate
- **Convergence**: The estimates should approach the exact ground state energy  
- **Diminishing returns**: Later Krylov dimensions provide smaller improvements

The exact ground state energy for our selected Hamiltonian was computed earlier via classical exact diagonalization and will be used as the reference for comparison with SKQD results.


In [ ]:
plot_skqd_convergence(csr_energies, lo_energies, range(1, max_k), exact_ground_state_energy)

## Postprocessing Acceleration: CSR matrix approach, single GPU vs CPU 

The data in the plot below was gathered by averaging over 5 runs on a NVIDIA H100 GPU and an Intel Xeon Platinum 8480CL CPU with 224 threads. The only thing that changed between the 2 data points was `vectorized_pj_on_gpu = True` and `vectorized_pj_on_gpu = False`

This substantial acceleration comes from:
- **Parallel Pauli string evaluation** across thousands of basis states simultaneously
- **Vectorized matrix element computation** leveraging CUDA cores
- **GPU-optimized sparse linear algebra** via cuSPARSE library for eigenvalue solving

The speedup becomes increasingly critical as the Krylov dimension grows, since the computational subspace dimension (and thus the matrix size) scales exponentially with k. For higher k values, GPU acceleration transforms previously intractable postprocessing into feasible computation times.


<img src="images/speedup.png" width="500">


## Postprocessing Scale-Up and Scale-Out: Linear Operator Approach, Multi-GPU Multi-Node

The plots below were generated using `distributed_eigsh()` with the Hamiltonian of an active site of a GPCR (G protein-coupled receptor) molecule. This demonstrates scaling behavior on a problem with a much larger computational subspace than the 22-qubit demo above.

<div style="display: flex; gap: 10px;">
  <img src="images/strong_scaling_timing.jpeg" width="450">
  <img src="images/efficiency.jpeg" width="450">
</div>

### Saving Hamiltonian Data

To run the multi-GPU solver, we first save the Hamiltonian data (pauli words, coefficients) and the sampled basis states so they can be loaded by each MPI rank.

In [ ]:
# # uncomment this to save the data

# cumulative_subspace_results = cumulative_results[max_k-1]
# basis_states = get_basis_states_as_array(cumulative_subspace_results, num_spins)

# with open('skqd_data.pkl', 'wb') as f:
#     pickle.dump({
#         'basis_states': basis_states,
#         'pauli_words': pauli_words,
#         'hamiltonian_coefficients': hamiltonian_coefficients_numpy,
#     }, f)


### Running the Distributed Solver

Running the cell below saves the code as `distributed_postprocessing.py` (it needs GPUs and `mpi4py`, so it is not run in the notebook). Launch it with MPI. For example, on a system with 8 GPUs:

```bash
mpirun -np 8 --bind-to none --allow-run-as-root python distributed_postprocessing.py
```

In [ ]:
%%writefile distributed_postprocessing.py
"""
Multi-GPU SKQD Eigenvalue Solver

Usage:
    Save the code below in a distributed_postprocessing.py file
    Execute with the command below, for example on a system with 8 GPUs: 
    mpirun -np 8 --bind-to none --allow-run-as-root python distributed_postprocessing.py 
"""

import numpy as np
import cupy as cp
from skqd_src.pre_and_postprocessing import *
from mpi4py import MPI
import pickle
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

n_devices = cp.cuda.runtime.getDeviceCount()
my_gpu = rank % n_devices
cp.cuda.Device(my_gpu).use()

tile_size = 32

#uncomment this to load the data
# with open('skqd_data.pkl', 'rb') as f:
#     data = pickle.load(f)
#     basis_states = data['basis_states']
#     pauli_words = data['pauli_words']
#     hamiltonian_coefficients_numpy = data['hamiltonian_coefficients']



subspace_dimension = basis_states.shape[0]

ham_data = prepare_hamiltonian_data(basis_states, pauli_words, hamiltonian_coefficients_numpy)
cp.cuda.Device(my_gpu).synchronize()

nccl_comm_obj = None
if size > 1:
    from cupy.cuda import nccl
    if rank == 0:
        nccl_id = nccl.get_unique_id()
    else:
        nccl_id = None
    nccl_id = comm.bcast(nccl_id, root=0)
    nccl_comm_obj = nccl.NcclCommunicator(size, nccl_id, rank)

np.random.seed(42)
v0_np = np.random.random((subspace_dimension,)).astype(np.float64)
v0_np = v0_np / np.linalg.norm(v0_np)
v0 = cp.asarray(v0_np, dtype=cp.complex128)

cp.get_default_memory_pool().free_all_blocks()

ncv = min(20, subspace_dimension - 1)

eigenvalues, = distributed_eigsh(
    ham_data, rank, size, nccl_comm_obj,
    k=1, which='SA', ncv=ncv, v0_full=v0,
    tile_size=tile_size, return_eigenvectors=False)

eigenvalue = eigenvalues[0]
cp.cuda.Device(my_gpu).synchronize()
eigenvalue_real = float(cp.asnumpy(eigenvalue).real)

MPI.Finalize()


## Summary

In this notebook we demonstrated the **Sample-Based Krylov Quantum Diagonalization (SKQD)** algorithm using CUDA-Q:

1. **Quantum phase**: Generated a sequence of Krylov states $U^k|\psi_0\rangle$ via qDRIFT time evolution on the 14-qubit Jordan-Wigner Hamiltonian of H$_2$O (STO-3G), and sampled each state in the computational basis.

2. **Classical post-processing**: Projected the Hamiltonian onto the computational subspace defined by the measurement outcomes, and extracted the ground state energy via eigenvalue solvers.

3. **Two solver approaches**: Compared an explicit GPU-accelerated CSR sparse matrix solver with a matrix-free Lanczos iteration (`distributed_eigsh`), showing both converge to the same result.

4. **GPU acceleration**: Demonstrated significant speedup of the classical post-processing when using GPU-vectorized operations compared to CPU.

5. **Multi-GPU scaling**: Showed how the matrix-free Lanczos solver scales across multiple GPUs using NCCL for distributed computation.
